In [1]:
import cv2
import numpy as np


def change_tool(image1, image2, threshold_value=30, min_area=5):
    """
    Compare two RGB/BGR images and return changed regions as GeoJSON.
    """

    # Convert images to grayscale
    gray1 = cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY)

    # Structural difference
    diff = cv2.absdiff(gray1, gray2)

    # Threshold
    _, change_mask = cv2.threshold(
        diff,
        threshold_value,
        255,
        cv2.THRESH_BINARY
    )

    # Find changed regions
    contours, _ = cv2.findContours(
        change_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    features = []

    for contour in contours:

        area = cv2.contourArea(contour)

        # Remove tiny noise
        if area < min_area:
            continue

        points = contour.reshape(-1, 2).tolist()

        # Close polygon
        if points[0] != points[-1]:
            points.append(points[0])

        features.append({
            "type": "Feature",
            "properties": {
                "area": float(area)
            },
            "geometry": {
                "type": "Polygon",
                "coordinates": [points]
            }
        })

    return {
        "type": "FeatureCollection",
        "features": features
    }

In [2]:
import os
import cv2
import numpy as np


def preprocess_cdvqa(before_dir, after_dir, output_dir):
    """
    Load matching Before/After CDVQA images and
    stitch them side-by-side into a single 2D array.
    """

    os.makedirs(output_dir, exist_ok=True)

    stitched_images = {}

    before_files = sorted(os.listdir(before_dir))

    for filename in before_files:

        before_path = os.path.join(before_dir, filename)
        after_path = os.path.join(after_dir, filename)

        # Make sure matching After image exists
        if not os.path.exists(after_path):
            continue

        before = cv2.imread(before_path)
        after = cv2.imread(after_path)

        if before is None or after is None:
            continue

        # Convert to grayscale
        before_gray = cv2.cvtColor(
            before,
            cv2.COLOR_BGR2GRAY
        )

        after_gray = cv2.cvtColor(
            after,
            cv2.COLOR_BGR2GRAY
        )

        # Make heights equal
        if before_gray.shape[0] != after_gray.shape[0]:

            height = min(
                before_gray.shape[0],
                after_gray.shape[0]
            )

            before_gray = cv2.resize(
                before_gray,
                (before_gray.shape[1], height)
            )

            after_gray = cv2.resize(
                after_gray,
                (after_gray.shape[1], height)
            )

        # Before | After
        stitched = np.hstack(
            (before_gray, after_gray)
        )

        stitched_images[filename] = stitched

        # Save
        cv2.imwrite(
            os.path.join(output_dir, filename),
            stitched
        )

    return stitched_images

In [3]:
def format_cdvqa_prompts(questions, answers):
    """
    Convert CDVQA question-answer data into
    strict conversational prompts.
    """

    prompts = []

    for question, answer in zip(questions, answers):

        prompt = {
            "role": "user",
            "content": (
                "Compare the left and right halves of this image. "
                "What changed?"
            )
        }

        prompts.append({
            "prompt": prompt,
            "answer": answer
        })

    return prompts

In [4]:
import os
print(os.listdir('/kaggle/input'))

['datasets', 'notebooks']


In [5]:
path1='/kaggle/input'
print(os.listdir(path1+'/datasets'))

['soumikrakshit']


In [6]:
os.listdir('/kaggle/input/datasets/soumikrakshit')

['onera-satellite-change-detection-dataset']

In [7]:
os.listdir('/kaggle/input/datasets/soumikrakshit/onera-satellite-change-detection-dataset')

['images', 'train_labels']

In [8]:
os.listdir('/kaggle/input/datasets/soumikrakshit/onera-satellite-change-detection-dataset/images')

['Onera Satellite Change Detection dataset - Images']

In [9]:
os.listdir('/kaggle/input/datasets/soumikrakshit/onera-satellite-change-detection-dataset/train_labels')

['Onera Satellite Change Detection dataset - Train Labels']

In [10]:
os.listdir('/kaggle/input/datasets/soumikrakshit/onera-satellite-change-detection-dataset/train_labels/Onera Satellite Change Detection dataset - Train Labels')

['paris',
 'rennes',
 'bercy',
 'abudhabi',
 'hongkong',
 'aguasclaras',
 'saclay_e',
 'README.txt',
 'beirut',
 'mumbai',
 'pisa',
 'beihai',
 'bordeaux',
 'cupertino',
 'nantes']

In [11]:
os.listdir('/kaggle/input/datasets/soumikrakshit/onera-satellite-change-detection-dataset/images/Onera Satellite Change Detection dataset - Images')

['paris',
 'rennes',
 'bercy',
 'abudhabi',
 'hongkong',
 'brasilia',
 'aguasclaras',
 'rio',
 'test.txt',
 'norcia',
 'valencia',
 'saclay_e',
 'montpellier',
 'lasvegas',
 'README.txt',
 'beirut',
 'mumbai',
 'train.txt',
 'all.txt',
 'pisa',
 'milano',
 'beihai',
 'saclay_w',
 'bordeaux',
 'dubai',
 'chongqing',
 'cupertino',
 'nantes']

In [12]:
images_base = "/kaggle/input/datasets/soumikrakshit/onera-satellite-change-detection-dataset/images/Onera Satellite Change Detection dataset - Images"

labels_base = "/kaggle/input/datasets/soumikrakshit/onera-satellite-change-detection-dataset/train_labels/Onera Satellite Change Detection dataset - Train Labels"

print("IMAGES:")
print(os.listdir(images_base + "/paris"))

print("\nLABELS:")
print(os.listdir(labels_base + "/paris"))

IMAGES:
['dates.txt', 'imgs_1_rect', 'imgs_2', 'paris.geojson', 'pair', 'imgs_2_rect', 'imgs_1']

LABELS:
['cm']


In [13]:
os.listdir(images_base + "/paris/imgs_1")

['S2A_OPER_MSI_L1C_TL_MPS__20161130T130757_A007527_T31UDQ_B08.tif',
 'S2A_OPER_MSI_L1C_TL_MPS__20161130T130757_A007527_T31UDQ_B01.tif',
 'S2A_OPER_MSI_L1C_TL_MPS__20161130T130757_A007527_T31UDQ_B11.tif',
 'S2A_OPER_MSI_L1C_TL_MPS__20161130T130757_A007527_T31UDQ_B12.tif',
 'S2A_OPER_MSI_L1C_TL_MPS__20161130T130757_A007527_T31UDQ_B02.tif',
 'S2A_OPER_MSI_L1C_TL_MPS__20161130T130757_A007527_T31UDQ_B09.tif',
 'S2A_OPER_MSI_L1C_TL_MPS__20161130T130757_A007527_T31UDQ_B06.tif',
 'S2A_OPER_MSI_L1C_TL_MPS__20161130T130757_A007527_T31UDQ_B04.tif',
 'S2A_OPER_MSI_L1C_TL_MPS__20161130T130757_A007527_T31UDQ_B05.tif',
 'S2A_OPER_MSI_L1C_TL_MPS__20161130T130757_A007527_T31UDQ_B03.tif',
 'S2A_OPER_MSI_L1C_TL_MPS__20161130T130757_A007527_T31UDQ_B07.tif',
 'S2A_OPER_MSI_L1C_TL_MPS__20161130T130757_A007527_T31UDQ_B8A.tif',
 'S2A_OPER_MSI_L1C_TL_MPS__20161130T130757_A007527_T31UDQ_B10.tif']

In [14]:
os.listdir(images_base + "/paris/imgs_2")

['T31UDQ_20171107T105229_B12.tif',
 'T31UDQ_20171107T105229_B03.tif',
 'T31UDQ_20171107T105229_B05.tif',
 'T31UDQ_20171107T105229_B11.tif',
 'T31UDQ_20171107T105229_B04.tif',
 'T31UDQ_20171107T105229_B01.tif',
 'T31UDQ_20171107T105229_B07.tif',
 'T31UDQ_20171107T105229_B08.tif',
 'T31UDQ_20171107T105229_B02.tif',
 'T31UDQ_20171107T105229_B06.tif',
 'T31UDQ_20171107T105229_B10.tif',
 'T31UDQ_20171107T105229_B8A.tif',
 'T31UDQ_20171107T105229_B09.tif']

In [15]:
os.listdir(labels_base + "/paris")

['cm']

In [16]:
os.listdir(labels_base + "/paris/cm")

['cm.png', 'paris-cm.tif']

In [17]:
print(os.listdir(images_base + "/paris/imgs_1_rect"))

['B05.tif', 'B09.tif', 'B07.tif', 'B11.tif', 'B04.tif', 'B01.tif', 'B06.tif', 'B03.tif', 'B8A.tif', 'B08.tif', 'B12.tif', 'B02.tif', 'B10.tif']


In [18]:
print(os.listdir(images_base + "/paris/imgs_2_rect"))

['B05.tif', 'B09.tif', 'B07.tif', 'B11.tif', 'B04.tif', 'B01.tif', 'B06.tif', 'B03.tif', 'B8A.tif', 'B08.tif', 'B12.tif', 'B02.tif', 'B10.tif']


In [19]:
import cv2

cm_png = cv2.imread(
    labels_base + "/paris/cm/cm.png",
    cv2.IMREAD_GRAYSCALE
)

cm_tif = cv2.imread(
    labels_base + "/paris/cm/paris-cm.tif",
    cv2.IMREAD_GRAYSCALE
)

print("cm.png:", cm_png.shape)
print("paris-cm.tif:", cm_tif.shape)

cm.png: (408, 390)
paris-cm.tif: (408, 390)


[ WARN:0@0.559] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 42112 (0xa480) encountered


In [20]:
import os
import cv2
import numpy as np
import shutil

# --------------------------------------------------
# Paths
# --------------------------------------------------

images_base = (
    "/kaggle/input/datasets/soumikrakshit/"
    "onera-satellite-change-detection-dataset/"
    "images/Onera Satellite Change Detection dataset - Images"
)

labels_base = (
    "/kaggle/input/datasets/soumikrakshit/"
    "onera-satellite-change-detection-dataset/"
    "train_labels/Onera Satellite Change Detection dataset - Train Labels"
)

output_base = "/kaggle/working/onera_test"

before_dir = os.path.join(output_base, "Before")
after_dir = os.path.join(output_base, "After")
labels_dir = os.path.join(output_base, "Labels")

os.makedirs(before_dir, exist_ok=True)
os.makedirs(after_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)


# --------------------------------------------------
# Function to create RGB from Sentinel bands
# --------------------------------------------------

def create_rgb(city_dir):
    
    red = cv2.imread(
        os.path.join(city_dir, "B04.tif"),
        cv2.IMREAD_UNCHANGED
    )

    green = cv2.imread(
        os.path.join(city_dir, "B03.tif"),
        cv2.IMREAD_UNCHANGED
    )

    blue = cv2.imread(
        os.path.join(city_dir, "B02.tif"),
        cv2.IMREAD_UNCHANGED
    )

    if red is None or green is None or blue is None:
        raise ValueError(f"Could not read RGB bands in {city_dir}")

    # Normalize each band to 0-255
    def normalize_band(band):
        band = band.astype(np.float32)

        min_val = band.min()
        max_val = band.max()

        if max_val == min_val:
            return np.zeros_like(band, dtype=np.uint8)

        band = (band - min_val) / (max_val - min_val)
        band = (band * 255).astype(np.uint8)

        return band

    red = normalize_band(red)
    green = normalize_band(green)
    blue = normalize_band(blue)

    # OpenCV uses BGR ordering
    rgb = cv2.merge([blue, green, red])

    return rgb


# --------------------------------------------------
# Select a few real cities
# --------------------------------------------------

cities = [
    "paris",
    "rennes",
    "bercy",
    "abudhabi",
    "hongkong"
]


# --------------------------------------------------
# Create adapted dataset
# --------------------------------------------------

for city in cities:

    before_source = os.path.join(
        images_base, city, "imgs_1_rect"
    )

    after_source = os.path.join(
        images_base, city, "imgs_2_rect"
    )

    label_source = os.path.join(
        labels_base, city, "cm"
    )

    # Create RGB Before
    before_rgb = create_rgb(before_source)

    # Create RGB After
    after_rgb = create_rgb(after_source)

    # Save them
    cv2.imwrite(
        os.path.join(before_dir, city + ".png"),
        before_rgb
    )

    cv2.imwrite(
        os.path.join(after_dir, city + ".png"),
        after_rgb
    )

    # Use cm.png as ground truth
    label_path = os.path.join(
        label_source,
        "cm.png"
    )

    if os.path.exists(label_path):

        shutil.copy2(
            label_path,
            os.path.join(
                labels_dir,
                city + ".png"
            )
        )

    print(
        city,
        "Before:", before_rgb.shape,
        "After:", after_rgb.shape
    )


print("\nDataset prepared!")
print("Before:", os.listdir(before_dir))
print("After :", os.listdir(after_dir))
print("Labels:", os.listdir(labels_dir))

paris Before: (408, 390, 3) After: (408, 390, 3)
rennes Before: (339, 563, 3) After: (339, 563, 3)
bercy Before: (395, 360, 3) After: (395, 360, 3)
abudhabi Before: (799, 785, 3) After: (799, 785, 3)
hongkong Before: (695, 540, 3) After: (695, 540, 3)

Dataset prepared!
Before: ['hongkong.png', 'abudhabi.png', 'paris.png', 'bercy.png', 'rennes.png']
After : ['hongkong.png', 'abudhabi.png', 'paris.png', 'bercy.png', 'rennes.png']
Labels: ['hongkong.png', 'abudhabi.png', 'paris.png', 'bercy.png', 'rennes.png']


In [21]:
for folder in [before_dir, after_dir, labels_dir]:
    for filename in os.listdir(folder):
        if filename.startswith("paris_"):
            os.remove(os.path.join(folder, filename))

In [22]:
print("Before:", sorted(os.listdir(before_dir)))
print("After:", sorted(os.listdir(after_dir)))
print("Labels:", sorted(os.listdir(labels_dir)))

Before: ['abudhabi.png', 'bercy.png', 'hongkong.png', 'paris.png', 'rennes.png']
After: ['abudhabi.png', 'bercy.png', 'hongkong.png', 'paris.png', 'rennes.png']
Labels: ['abudhabi.png', 'bercy.png', 'hongkong.png', 'paris.png', 'rennes.png']


In [23]:
before = cv2.imread(
    os.path.join(before_dir, "paris.png")
)

after = cv2.imread(
    os.path.join(after_dir, "paris.png")
)

result = change_tool(before, after)

print("Number of detected changes:",
      len(result["features"]))

Number of detected changes: 35


In [24]:
print(result)

{'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'properties': {'area': 9.0}, 'geometry': {'type': 'Polygon', 'coordinates': [[[88, 372], [88, 374], [87, 375], [84, 375], [82, 373], [84, 375], [84, 377], [87, 377], [87, 375], [88, 374], [89, 374], [90, 373], [89, 372], [88, 372]]]}}, {'type': 'Feature', 'properties': {'area': 67.0}, 'geometry': {'type': 'Polygon', 'coordinates': [[[62, 356], [61, 357], [60, 357], [60, 360], [59, 361], [58, 361], [59, 362], [58, 363], [59, 364], [61, 364], [61, 363], [60, 362], [61, 361], [62, 361], [64, 363], [65, 363], [66, 362], [68, 362], [69, 363], [68, 364], [69, 364], [70, 365], [70, 364], [72, 362], [74, 362], [75, 363], [74, 364], [73, 364], [73, 365], [75, 365], [76, 366], [74, 368], [73, 368], [76, 368], [77, 367], [76, 366], [76, 365], [75, 364], [76, 363], [75, 363], [74, 362], [71, 362], [69, 360], [69, 359], [70, 358], [68, 358], [67, 359], [66, 358], [66, 356], [62, 356]]]}}, {'type': 'Feature', 'properties': {'area': 5.0},